# Batch Scripting the PhysIO Toolbox for Physiological Noise Modelling

**Author:** Kelly G. Garner, Michèle Masson-Trottier

<div style="line-height: 2;">
<a href="https://github.com/kel-github"><img src="https://img.shields.io/badge/-Kelly_G._Garner-181717?logo=github" alt="GitHub"></a><br>
<a href="https://github.com/micmas"><img src="https://img.shields.io/badge/-Michèle_Masson--Trottier-181717?logo=github" alt="GitHub"></a> <a href="https://orcid.org/0000-0002-0642-5662"><img src="https://img.shields.io/badge/ORCID-0000--0002--0642--5662-green?logo=orcid" alt="ORCID"></a>
</div>

**Date:** 30/03/2026

**License:**
<div style="margin-top: 10px;">
    <a href="https://opensource.org/licenses/MIT" target="_blank" style="color: #0066cc;">
        <i class="fas fa-balance-scale"></i> MIT License
    </a>
</div>

**Note:** If this notebook uses neuroimaging tools from Neurocontainers, those tools retain their original licenses. Please see <a href="https://neurodesk.org/overview/how-to-cite-us/" target="_blank" style="color: #0066cc;">Neurodesk citation guidelines</a> for details.

### Citation and Resources

**PhysIO Toolbox**
: Kasper, L., Bollmann, S., Diaconescu, A.O., Hutton, C., Heinzle, J., Iglesias, S., Hauser, T.U., Sebold, M., Manjaly, Z.-M., Pruessmann, K.P., Stephan, K.E., 2017. The PhysIO Toolbox for Modeling Physiological Noise in fMRI Data. *Journal of Neuroscience Methods*, 276, 56–72. https://doi.org/10.1016/j.jneumeth.2016.10.019

**SPM (Statistical Parametric Mapping)**
: Ashburner, J., Barnes, G., Chen, C., Daunizeau, J., Flandin, G., Friston, K., Kiebel, S., Kilner, J., Litvak, V., Moran, R., Penny, W., Phillips, C., Razi, A., Stephan, K., Wietzikoski, S., Worsley, K. (2014). SPM12 Manual. The FIL Methods Group.

**MATLAB**
: MATLAB Release 2023b. The MathWorks, Inc., Natick, Massachusetts, United States.

## Overview

PhysIO models physiological noise (cardiac, respiratory) in fMRI data. This example notebook creates a PhysIO batch script for a single participant and explains how to generalise it for multiple subjects.

**Requirements:**
- MATLAB license configured in Neurodesk (see MATLAB tutorial)
- Physiological recordings in `.log` format (BIDS-compatible)
- fMRI data in NIfTI format

## Load software tools

In [ ]:
import module
await module.load('matlab/R2023b')
await module.list()

## 1. Set Up Your Data Structure

Organise your data in BIDS format. PhysIO expects physiological recordings alongside the fMRI data:

In [ ]:
%%bash
# Example BIDS data structure for PhysIO
mkdir -p ~/neurodesktop-storage/physio_batch/
cat << 'EOF'
Expected BIDS structure:
  study/
    sub-01/
      func/
        sub-01_task-rest_bold.nii.gz
        sub-01_task-rest_physio.tsv.gz    ← physiological recordings
        sub-01_task-rest_physio.json      ← metadata (columns, sampling rate)
    sub-02/
      func/
        ...
EOF

## 2. Generate a PhysIO Batch Script

PhysIO batch scripts are MATLAB .m files. The best practice is to first generate an example script using the PhysIO GUI, then adapt it for batch processing.

Here we create the batch script programmatically using Python:

In [ ]:
import os

# Define paths - adjust to your actual data
base_dir = os.path.expanduser('~/neurodesktop-storage/physio_batch/')
subject = 'sub-01'
task = 'rest'

# Create output directory
os.makedirs(os.path.join(base_dir, subject), exist_ok=True)

# Write the PhysIO batch script
batch_script = f"""% PhysIO Batch Script - {subject}, task-{task}
% Generated by NeurodeskEDU physio_batch_workflow example

%% Initialize SPM
spm('defaults', 'FMRI');
spm_jobman('initcfg');

%% Define file paths
base_dir = '{base_dir}';
subject  = '{subject}';
task     = '{task}';

fmri_file = fullfile(base_dir, subject, 'func', ...
    [subject '_task-' task '_bold.nii.gz']);
physio_file = fullfile(base_dir, subject, 'func', ...
    [subject '_task-' task '_physio.tsv.gz']);
out_dir = fullfile(base_dir, subject, 'physio_output');
mkdir(out_dir);

%% Build PhysIO matlabbatch structure
matlabbatch{{1}}.spm.tools.physio.save_dir = {{out_dir}};
matlabbatch{{1}}.spm.tools.physio.log_files.vendor = 'BIDS';
matlabbatch{{1}}.spm.tools.physio.log_files.cardiac = {{physio_file}};
matlabbatch{{1}}.spm.tools.physio.log_files.respiration = {{physio_file}};
matlabbatch{{1}}.spm.tools.physio.log_files.scan_timing = {{}};
matlabbatch{{1}}.spm.tools.physio.log_files.sampling_interval = 0.01;  % 100 Hz
matlabbatch{{1}}.spm.tools.physio.log_files.relative_start_acquisition = 0;

%% Scan timing
matlabbatch{{1}}.spm.tools.physio.scan_timing.sqpar.Nslices = 36;
matlabbatch{{1}}.spm.tools.physio.scan_timing.sqpar.NslicesPerBeat = 0;
matlabbatch{{1}}.spm.tools.physio.scan_timing.sqpar.TR = 2.0;
matlabbatch{{1}}.spm.tools.physio.scan_timing.sqpar.Ndummies = 3;
matlabbatch{{1}}.spm.tools.physio.scan_timing.sqpar.Nscans = 200;

%% Preprocessing
matlabbatch{{1}}.spm.tools.physio.preproc.cardiac.modality = 'PPU';
matlabbatch{{1}}.spm.tools.physio.preproc.cardiac.filter.no = struct([]);
matlabbatch{{1}}.spm.tools.physio.preproc.cardiac.initial_cpulse_select.auto_matched.min = 0.4;
matlabbatch{{1}}.spm.tools.physio.preproc.cardiac.initial_cpulse_select.auto_matched.file = 'initial_cpulse_kRpeakfile.mat';

%% Noise model (RETROICOR)
matlabbatch{{1}}.spm.tools.physio.model.retroicor.yes.order.c = 3;
matlabbatch{{1}}.spm.tools.physio.model.retroicor.yes.order.r = 4;
matlabbatch{{1}}.spm.tools.physio.model.retroicor.yes.order.cr = 1;
matlabbatch{{1}}.spm.tools.physio.model.rvt.no = struct([]);
matlabbatch{{1}}.spm.tools.physio.model.hrv.no = struct([]);
matlabbatch{{1}}.spm.tools.physio.model.noise_rois.no = struct([]);
matlabbatch{{1}}.spm.tools.physio.model.movement.no = struct([]);
matlabbatch{{1}}.spm.tools.physio.model.other.no = struct([]);
matlabbatch{{1}}.spm.tools.physio.model.output_multiple_regressors = ...
    fullfile(out_dir, 'physio_regressors.txt');
matlabbatch{{1}}.spm.tools.physio.model.output_physio = ...
    fullfile(out_dir, 'physio_model.mat');

%% Verbose output
matlabbatch{{1}}.spm.tools.physio.verbose.level = 2;
matlabbatch{{1}}.spm.tools.physio.verbose.fig_output_file = ...
    fullfile(out_dir, 'physio_diagnostic.fig');
matlabbatch{{1}}.spm.tools.physio.verbose.use_tabs = false;

%% Run
spm_jobman('run', matlabbatch);
disp('PhysIO batch complete!');
"""

script_path = os.path.join(base_dir, f'run_physio_{subject}.m')
with open(script_path, 'w') as f:
    f.write(batch_script)

print(f"Script written to: {script_path}")
print(f"Script length: {len(batch_script.splitlines())} lines")

## 3. Generalise for Multiple Subjects

Wrap the script generation in a Python loop to create a batch script for each participant:

In [ ]:
import os
import glob

base_dir = os.path.expanduser('~/neurodesktop-storage/physio_batch/')

def create_physio_script(base_dir, subject, task='rest',
                          TR=2.0, Nscans=200, Ndummies=3, Nslices=36):
    """Generate a PhysIO batch .m script for one subject."""
    out_dir = os.path.join(base_dir, subject, 'physio_output')
    os.makedirs(out_dir, exist_ok=True)
    
    physio_file = os.path.join(base_dir, subject, 'func',
                               f'{subject}_task-{task}_physio.tsv.gz')
    
    script = f"""% PhysIO - {subject} task-{task}
spm('defaults', 'FMRI');
spm_jobman('initcfg');

out_dir = '{out_dir}';
matlabbatch{{1}}.spm.tools.physio.save_dir = {{out_dir}};
matlabbatch{{1}}.spm.tools.physio.log_files.vendor = 'BIDS';
matlabbatch{{1}}.spm.tools.physio.log_files.cardiac = {{'{physio_file}'}};
matlabbatch{{1}}.spm.tools.physio.log_files.respiration = {{'{physio_file}'}};
matlabbatch{{1}}.spm.tools.physio.log_files.sampling_interval = 0.01;
matlabbatch{{1}}.spm.tools.physio.scan_timing.sqpar.Nslices = {Nslices};
matlabbatch{{1}}.spm.tools.physio.scan_timing.sqpar.TR = {TR};
matlabbatch{{1}}.spm.tools.physio.scan_timing.sqpar.Ndummies = {Ndummies};
matlabbatch{{1}}.spm.tools.physio.scan_timing.sqpar.Nscans = {Nscans};
matlabbatch{{1}}.spm.tools.physio.model.retroicor.yes.order.c = 3;
matlabbatch{{1}}.spm.tools.physio.model.retroicor.yes.order.r = 4;
matlabbatch{{1}}.spm.tools.physio.model.retroicor.yes.order.cr = 1;
matlabbatch{{1}}.spm.tools.physio.model.output_multiple_regressors = ...
    fullfile(out_dir, 'physio_regressors.txt');
matlabbatch{{1}}.spm.tools.physio.verbose.level = 2;
spm_jobman('run', matlabbatch);
"""
    
    script_path = os.path.join(base_dir, f'run_physio_{subject}.m')
    with open(script_path, 'w') as f:
        f.write(script)
    return script_path

# Generate scripts for all subjects
subjects = [f'sub-{i:02d}' for i in range(1, 6)]  # sub-01 through sub-05
for sub in subjects:
    path = create_physio_script(base_dir, sub)
    print(f"Created: {path}")

## 4. Run PhysIO from Neurodesk

Once your MATLAB license is configured (see the MATLAB tutorial), run the batch script from the terminal:

```bash
# Load MATLAB
ml matlab/R2023b

# Run PhysIO for one subject
matlab -nodisplay -r "run('~/neurodesktop-storage/physio_batch/run_physio_sub-01.m'); exit"
```

Or loop over all subjects:

```bash
for sub in sub-01 sub-02 sub-03 sub-04 sub-05; do
    matlab -nodisplay -r "run('~/neurodesktop-storage/physio_batch/run_physio_${sub}.m'); exit"
done
```

In [ ]:
%%bash
# List the generated batch scripts
ls -lh ~/neurodesktop-storage/physio_batch/run_physio_*.m 2>/dev/null || \
    echo "Run the cells above first to generate the batch scripts"

In [ ]:
%%bash
# Preview the first script
head -40 ~/neurodesktop-storage/physio_batch/run_physio_sub-01.m 2>/dev/null || \
    echo "Script not found - run the generation cells above"

## 5. Check Outputs

After running PhysIO, check the output directory for regressors and diagnostic figures:

In [ ]:
%%bash
ls ~/neurodesktop-storage/physio_batch/sub-01/physio_output/ 2>/dev/null || \
    echo "No output yet - run PhysIO using the commands in Section 4"

The key output file is `physio_regressors.txt`, which contains the RETROICOR nuisance regressors. These can be included in your SPM or FSL first-level GLM as confounds.

## Dependencies in Jupyter/Python

In [ ]:
%load_ext watermark
%watermark
%watermark --iversions